# Python 网络机器人（Web Bot）入门指南

本 Notebook 系统介绍 Python 网络机器人的核心知识、工具栈和学习路径。

## 一、网络机器人的主要类型

| 类型 | 说明 | 典型场景 |
|------|------|----------|
| **网络爬虫** | 自动抓取网页内容并解析 | 数据采集、价格监控、新闻聚合 |
| **浏览器自动化** | 模拟真人操作浏览器 | 自动登录、表单填写、测试 |
| **API 机器人** | 直接调用后端接口 | 社交媒体机器人、消息推送 |
| **聊天机器人** | 基于 NLP 的自动对话 | 客服机器人、智能助手 |

## 二、Python 核心工具栈

### 1. 基础请求库
- **`requests`**：最经典的 HTTP 库，适合静态页面抓取
- **`httpx`**：支持异步 HTTP/2，性能更好，现代项目首选
- **`urllib`**：Python 内置，一般不建议直接使用

### 2. HTML 解析库
- **`BeautifulSoup`**（bs4）：最友好的 HTML/XML 解析器，适合初学者
- **`lxml`**：基于 C 语言，解析速度极快，支持 XPath
- **`pyquery`**：类似 jQuery 的语法风格

### 3. 浏览器自动化
- **`Selenium`**：老牌方案，支持多浏览器，社区庞大
- **`Playwright`**（推荐）：微软出品，支持异步、自动等待、多浏览器
- **`DrissionPage`**：国产框架，整合 requests 和浏览器控制

### 4. 异步与高性能框架
- **`aiohttp`**：异步 HTTP 客户端/服务器
- **`Scrapy`**：专业级爬虫框架，内置调度、去重、管道、中间件
- **`Celery`**：分布式任务队列，适合大规模爬虫调度

### 5. 反爬对抗工具
- **验证码识别**：`ddddocr`（开源）、打码平台 API
- **指纹伪装**：`selenium-stealth`、`playwright-stealth`
- **代理池**：自建或购买住宅代理/机房代理
- **JS 逆向**：配合浏览器 DevTools 分析加密逻辑

## 三、环境准备：安装常用库

In [ ]:
# 运行以下命令安装常用爬虫库（取消注释后执行）
# !pip install requests beautifulsoup4 lxml httpx playwright scrapy aiohttp

## 四、实战示例

### 示例 1：requests + BeautifulSoup 抓取静态页面

In [ ]:
import requests
from bs4 import BeautifulSoup

# 定义目标 URL
target_url = "https://quotes.toscrape.com/"

# 发送 GET 请求
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}
response = requests.get(target_url, headers=headers)

# 解析 HTML
soup = BeautifulSoup(response.text, "html.parser")

# 提取名言和作者
quotes = soup.find_all("div", class_="quote")
for quote in quotes[:5]:  # 只打印前 5 条
    text = quote.find("span", class_="text").get_text()
    author = quote.find("small", class_="author").get_text()
    print(f"{author}: {text}")
    print("-" * 50)

### 示例 2：lxml + XPath 解析

In [ ]:
import requests
from lxml import html

target_url = "https://www.tiobe.com/tiobe-index/"
response = requests.get(target_url, headers={
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
})

document = html.fromstring(response.text)

# 解析表头
headers = document.xpath("//table[@id='top20']//th//text()")
print("表头:", headers)

# 解析表格数据
rows = document.xpath("//table[@id='top20']//tr[position()>1]")
for row in rows[:5]:  # 只打印前 5 行
    cells = row.xpath("./td//text()")
    print(cells)

### 示例 3：Playwright 浏览器自动化（需先安装浏览器）

In [ ]:
# 先安装浏览器（仅需执行一次）
# !playwright install chromium

from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto("https://quotes.toscrape.com/js/")
    
    # 等待页面加载完成
    page.wait_for_selector(".quote")
    
    # 提取数据
    quotes = page.query_selector_all(".quote")
    for quote in quotes[:3]:
        text = quote.query_selector(".text").inner_text()
        author = quote.query_selector(".author").inner_text()
        print(f"{author}: {text}")
    
    browser.close()

### 示例 4：异步爬虫 aiohttp

In [ ]:
import asyncio
import aiohttp

async def fetch(url):
    async with aiohttp.ClientSession() as session:
        async with session.get(url) as response:
            return await response.text()

urls = [
    "https://httpbin.org/get?id=1",
    "https://httpbin.org/get?id=2",
    "https://httpbin.org/get?id=3"
]

# 并发请求
tasks = [fetch(url) for url in urls]
results = await asyncio.gather(*tasks)

for i, result in enumerate(results):
    print(f"请求 {i+1} 返回长度: {len(result)}")

## 五、一个完整的爬虫流程

```
1. 分析目标网站 → 确定数据来源（HTML/JSON/API）
2. 发送 HTTP 请求 → requests / httpx / Playwright
3. 解析响应内容 → BeautifulSoup / XPath / 正则
4. 存储数据 → CSV / JSON / MySQL / MongoDB
5. 处理反爬 → 代理 / 请求头 / 验证码 / 浏览器模拟
6. 调度与监控 → Scrapy / Celery / 定时任务
```

## 六、学习路径建议

### 第一阶段：基础入门（1-2 周）
1. 掌握 `requests` + `BeautifulSoup` 做简单静态页面抓取
2. 学习 XPath 或 CSS 选择器定位元素
3. 练习将数据保存到 CSV/JSON

**练手项目**：爬取豆瓣电影 Top250、天气网站、新闻标题

### 第二阶段：进阶实战（2-4 周）
1. 学习 `Scrapy` 框架，理解 Spider、Pipeline、Middleware
2. 掌握异步编程（`asyncio` + `aiohttp`）
3. 学习数据库操作（`pymongo`、`sqlalchemy`）

**练手项目**：构建一个可配置的多网站新闻聚合器

### 第三阶段：高阶对抗（持续）
1. 学习 `Playwright` 处理动态渲染页面（SPA、Ajax）
2. 了解常见反爬机制及应对策略
3. 学习 JS 逆向基础（断点调试、加密分析）
4. 搭建代理池和分布式爬虫架构

**练手项目**：模拟登录并抓取需要身份验证的数据

## 七、重要注意事项

### ⚠️ 法律与伦理边界
- **遵守 robots.txt**：网站根目录下的爬虫协议文件
- **控制请求频率**：避免对目标服务器造成过大压力（建议加延时、限制并发）
- **尊重版权与隐私**：不要抓取受版权保护的内容或个人隐私数据
- **了解相关法规**：《网络安全法》《数据安全法》《个人信息保护法》

### 技术最佳实践
- 优先使用**公开 API**，而非直接爬取网页
- 做好**异常处理**和**重试机制**
- 使用**日志记录**便于排查问题
- 数据存储时考虑**去重**（URL 去重、内容去重）

## 八、推荐学习资源

| 资源 | 说明 |
|------|------|
| **Scrapy 官方文档** | 最权威的框架指南 |
| **Playwright 官方文档** | 现代浏览器自动化的标准 |
| **《Python 网络数据采集》**（Ryan Mitchell） | 经典入门书籍 |
| **崔庆才《Python3 网络爬虫开发实战》** | 中文经典，案例丰富 |

---

祝你学习顺利！如有具体问题，欢迎继续交流。